## 1. Load Context Dataset

读取 `02_context_features.ipynb` 生成的上下文数据，作为人工标注的基础。

In [82]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    r"D:\JupyterProject\AIML339_Toxicity"
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chat_with_context.parquet"
)

df = pd.read_parquet(DATA_PATH)

print("Shape:")
print(df.shape)

display(df.head())

Shape:
(1361583, 20)


,match_id,time,player_slot,unit,key,prev_message_1,prev_message_2,prev_message_3,prev_message_4,prev_message_5,gold,xp,last_hits,messages_last_60s,repeated_message_60s,message_interval,player_team,seconds_since_teamfight,recent_teamfight_120s,recent_teamfight_loss_120s
0,0,1524,0,Double T,fuck my ass,is cruel,fate,hah,bye,mvp ulti,6294,5841,15,0,0,NaN,radiant,196.0,0,0
1,0,1721,0,Double T,ka bu tooooooooooooo,fuck my ass,is cruel,fate,hah,bye,7564,7747,20,0,0,197.0,radiant,12.0,1,1
2,0,2387,0,Double T,6k slayer,gG,wahaha,really ?,torso highest here,wlwlwlwllww,13409,13356,30,0,0,666.0,radiant,103.0,1,0
3,0,2388,0,Double T,haha,6k slayer,gG,wahaha,really ?,torso highest here,13409,13356,30,1,0,1.0,radiant,0.0,1,0
4,0,5,1,Monkey,space created,None,None,None,None,None,0,0,0,0,0,NaN,radiant,NaN,0,0


## 2. Clean Missing Messages

移除没有聊天文本的记录，因为这些消息无法进行人工标注。

In [83]:
print(
    "Missing messages:",
    df["key"].isna().sum()
)

label_data = (
    df[
        df["key"].notna()
    ]
    .copy()
)

label_data["key"] = (
    label_data["key"]
    .astype(str)
    .str.strip()
)

label_data = label_data[
    label_data["key"] != ""
].copy()

print(
    "Rows after cleaning:",
    len(label_data)
)

Missing messages: 13
Rows after cleaning: 1361381


## 3. Labeling Rules

### Class 0 — Normal Communication

Normal tactical communication, cooperation requests, questions, game information, or general social conversation.

Examples:

- "group mid"
- "wait"
- "roshan?"
- "nice"
- "gg"

### Class 1 — Limited Emotional Expression

Brief frustration, complaints, negative reactions, or emotional expressions without serious personal attacks, threats, discrimination, or persistent harassment.

Examples:

- "come on guys"
- "this is frustrating"
- "wtf"
- "why always me"
- "this team is throwing"

Context should be considered when the meaning is ambiguous.

### Class 2 — Harmful Behaviour

Direct insults, serious personal attacks, threats, discriminatory language, or repeated targeted harassment.

Examples:

- "you are trash"
- "uninstall"
- direct threats
- discriminatory attacks
- repeated harassment of the same player

Explicit threats, discrimination, and serious insults remain Harmful even if the player is frustrated by the match.

## 4. Build Readable Context

将前五条聊天消息组合成一个可读的文本字段，方便人工标注时理解当前消息的聊天背景。

In [84]:
previous_columns = [
    "prev_message_5",
    "prev_message_4",
    "prev_message_3",
    "prev_message_2",
    "prev_message_1"
]


def build_context(row):
    messages = []

    for col in previous_columns:
        value = row[col]

        if pd.notna(value):
            messages.append(str(value))

    return " | ".join(messages)


label_data["chat_context"] = (
    label_data.apply(
        build_context,
        axis=1
    )
)

display(
    label_data[
        [
            "key",
            "chat_context",
            "messages_last_60s",
            "repeated_message_60s",
            "recent_teamfight_loss_120s"
        ]
    ].head(20)
)

,key,chat_context,messages_last_60s,repeated_message_60s,recent_teamfight_loss_120s
0,fuck my ass,mvp ulti | bye | hah | fate | is cruel,0,0,0
1,ka bu tooooooooooooo,bye | hah | fate | is cruel | fuck my ass,0,0,1
2,6k slayer,wlwlwlwllww | torso highest here | really ? | ...,0,0,0
3,haha,torso highest here | really ? | wahaha | gG | ...,1,0,0
4,space created,,0,0,0
5,hah,space created,1,0,0
6,hah,space created | hah | ez 500 | mvp ulti | bye,0,0,0
7,wtf,hah | fate | is cruel | fuck my ass | ka bu to...,0,0,0
8,TA?,fate | is cruel | fuck my ass | ka bu tooooooo...,1,0,0
9,u srsly?,is cruel | fuck my ass | ka bu tooooooooooooo ...,2,0,0


## 5. Create Pilot Candidate Groups

从完整数据中建立不同类型的候选样本，包括随机聊天、情绪表达候选和有害内容候选。

这些规则只用于抽样，不直接作为最终标签。

In [85]:
import re

# Simple terms used only for sampling
emotional_pattern = re.compile(
    r"\b("
    r"wtf|why|bad|sad|throw|throwing|"
    r"come on|annoy|frustrat|lost|lose"
    r")\b",
    flags=re.IGNORECASE
)

harmful_pattern = re.compile(
    r"\b("
    r"fuck|shit|trash|idiot|stupid|"
    r"noob|uninstall|retard|kill yourself|kys"
    r")\b",
    flags=re.IGNORECASE
)

label_data["emotional_candidate"] = (
    label_data["key"]
    .str.contains(emotional_pattern, na=False)
)

label_data["harmful_candidate"] = (
    label_data["key"]
    .str.contains(harmful_pattern, na=False)
)

print("Emotional candidates:",
      label_data["emotional_candidate"].sum())

print("Harmful candidates:",
      label_data["harmful_candidate"].sum())

C:\Users\OEM\AppData\Local\Temp\ipykernel_29672\709238071.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  label_data["key"]
C:\Users\OEM\AppData\Local\Temp\ipykernel_29672\709238071.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  label_data["key"]


Emotional candidates: 33496
Harmful candidates: 46447


## 6. Create Pilot Sample

建立 300 条人工标注样本，包括随机消息、情绪表达候选和有害内容候选，以测试三分类标注规则。

In [86]:
RANDOM_STATE = 339

# 150 completely random messages
random_sample = (
    label_data
    .sample(
        n=150,
        random_state=RANDOM_STATE
    )
    .copy()
)

random_sample["sampling_group"] = "random"


# Avoid selecting messages already in random sample
remaining = label_data.drop(
    index=random_sample.index
)

# 75 emotional candidates
emotional_pool = remaining[
    remaining["emotional_candidate"]
    &
    ~remaining["harmful_candidate"]
]

emotional_sample = (
    emotional_pool
    .sample(
        n=min(75, len(emotional_pool)),
        random_state=RANDOM_STATE
    )
    .copy()
)

emotional_sample["sampling_group"] = "emotional_candidate"


# Remove already selected messages again
remaining = remaining.drop(
    index=emotional_sample.index
)

# 75 harmful candidates
harmful_pool = remaining[
    remaining["harmful_candidate"]
]

harmful_sample = (
    harmful_pool
    .sample(
        n=min(75, len(harmful_pool)),
        random_state=RANDOM_STATE
    )
    .copy()
)

harmful_sample["sampling_group"] = "harmful_candidate"


pilot = pd.concat(
    [
        random_sample,
        emotional_sample,
        harmful_sample
    ],
    ignore_index=True
)

# Shuffle the order so the sampling group does not influence labeling
pilot = (
    pilot
    .sample(
        frac=1,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)

pilot["annotation_id"] = range(
    1,
    len(pilot) + 1
)

print("Pilot size:")
print(len(pilot))

print("\nSampling groups:")
print(pilot["sampling_group"].value_counts())

Pilot size:
300

Sampling groups:
sampling_group
random                 150
harmful_candidate       75
emotional_candidate     75
Name: count, dtype: int64


## 7. Prepare Manual Annotation File

整理人工标注需要查看的消息、聊天历史和主要上下文特征，并添加空白标签列。

In [87]:
annotation_columns = [
    "annotation_id",
    "match_id",
    "time",
    "key",
    "chat_context",

    "messages_last_60s",
    "repeated_message_60s",
    "message_interval",

    "gold",
    "xp",
    "last_hits",

    "recent_teamfight_120s",
    "recent_teamfight_loss_120s",

    "sampling_group"
]

annotation_df = (
    pilot[annotation_columns]
    .copy()
)

# Blank columns for manual annotation
annotation_df["label"] = ""
annotation_df["notes"] = ""

display(annotation_df.head(20))

,annotation_id,match_id,time,key,chat_context,messages_last_60s,repeated_message_60s,message_interval,gold,xp,last_hits,recent_teamfight_120s,recent_teamfight_loss_120s,sampling_group,label,notes
0,1,24301,1060,man ur life must be shit,retard | if u didnt suck so much we coudla pla...,0,0,NaN,10280,6610,93,0,0,harmful_candidate,,
1,2,49193,2180,dont know why,steo by step | we will | relax | fuck valve | ...,1,0,60.0,13984,17394,91,1,1,emotional_candidate,,
2,3,4367,2072,can u end,im sure he proved to be a real challenge mid k...,1,0,67.0,13532,16118,166,1,1,random,,
3,4,1909,2191,noob jugger,how did you get those mmr storm | account buye...,2,0,4.0,17479,18522,80,1,0,harmful_candidate,,
4,5,16069,1889,it is not about how bad he is,win lol | hehe | dp won | i tried to win he tr...,2,0,54.0,18559,19904,140,1,0,emotional_candidate,,
5,6,31390,1606,bara,bara retarded | gg | report pls blood | ez | i...,1,0,2.0,10773,12813,151,1,0,random,,
6,7,34912,3769,yes storm was bad,be positive | fk urslrf | are you ready? | fki...,4,0,7.0,23156,32783,159,1,0,emotional_candidate,,
7,8,3381,1923,noob am,i dont know whats top and mid | mid no impact ...,1,0,1.0,11685,11156,47,1,0,harmful_candidate,,
8,9,7618,2740,wtf is that,GIVE UP | 1 bloodstone no aghs lina vs huskar ...,2,0,3.0,17843,21311,100,0,0,emotional_candidate,,
9,10,40321,2281,sorry jugg,Gg | lol | gg | ez | gg,5,0,2.0,11834,14965,90,1,0,random,,


## 8. Export Pilot Annotation File

将 300 条 pilot sample 导出为 CSV，方便进行人工三分类标注。

In [88]:
LABEL_DIR = (
    PROJECT_ROOT
    / "data"
    / "labels"
)

LABEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

pilot_path = (
    LABEL_DIR
    / "pilot_annotation_300.csv"
)

annotation_df.to_csv(
    pilot_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved to:")
print(pilot_path)

Saved to:
D:\JupyterProject\AIML339_Toxicity\data\labels\pilot_annotation_300.csv


In [89]:
print("Label folder:")
print(PROJECT_ROOT / "data" / "labels")

print("\nCSV files:")
for file in (PROJECT_ROOT / "data" / "labels").glob("*.csv"):
    print(file.name)

Label folder:
D:\JupyterProject\AIML339_Toxicity\data\labels

CSV files:
annotation_additional_900.csv
annotation_additional_900_labeled.csv
pilot_annotation_300.csv
pilot_annotation_300_labeled.csv


## 9. Load Reviewed Labels

读取完成标注的 pilot 数据，并移除保存 CSV 时产生的空白行。

In [90]:
LABEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "labels"
    / "pilot_annotation_300_labeled.csv"
)

reviewed_labels = pd.read_csv(
    LABEL_PATH
)

print("Rows before cleaning:")
print(len(reviewed_labels))

reviewed_labels = (
    reviewed_labels[
        reviewed_labels["annotation_id"].notna()
    ]
    .copy()
)

reviewed_labels["label"] = (
    reviewed_labels["label"]
    .astype(int)
)

print("\nRows after cleaning:")
print(len(reviewed_labels))

print("\nLabel distribution:")
print(
    reviewed_labels["label"]
    .value_counts()
    .sort_index()
)

Rows before cleaning:
300

Rows after cleaning:
300

Label distribution:
label
0    117
1    118
2     65
Name: count, dtype: int64


## 10. Check Label Completeness

确认所有 pilot 样本都已经完成三分类标注。

In [91]:
print(
    "Missing labels:",
    reviewed_labels["label"].isna().sum()
)

print(
    "Valid labels only:",
    reviewed_labels["label"]
    .isin([0, 1, 2])
    .all()
)

Missing labels: 0
Valid labels only: True


## 11. Check Label Distribution

检查三类标签的数量和比例，了解当前 pilot 数据的类别分布。

In [92]:
label_counts = (
    reviewed_labels["label"]
    .value_counts()
    .sort_index()
)

label_percent = (
    reviewed_labels["label"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

distribution = pd.DataFrame({
    "count": label_counts,
    "percentage": label_percent.round(2)
})

distribution.index = [
    "Normal",
    "Limited Emotional",
    "Harmful"
]

display(distribution)

,count,percentage
Normal,117,39.00
Limited Emotional,118,39.33
Harmful,65,21.67


## 12. Check Sampling Groups

比较不同抽样组和最终人工标签之间的关系，确认候选规则只是帮助抽样，而不是直接决定标签。

In [93]:
sampling_check = pd.crosstab(
    reviewed_labels["sampling_group"],
    reviewed_labels["label"]
)

sampling_check.columns = [
    "Normal",
    "Limited Emotional",
    "Harmful"
]

display(sampling_check)

,Normal,Limited Emotional,Harmful
sampling_group,,,
emotional_candidate,8,62,5
harmful_candidate,4,22,49
random,105,34,11


## 13. Create Additional Annotation Sample

根据 pilot 的结果继续抽取 900 条新消息，用于建立正式训练数据。

新样本包括随机消息、情绪候选和有害候选，同时避免与 pilot 样本重复。

In [94]:
print(reviewed_labels.columns.tolist())

display(
    reviewed_labels[
        [
            "annotation_id",
            "match_id",
            "time",
            "key"
        ]
    ].head()
)

['annotation_id', 'match_id', 'time', 'key', 'chat_context', 'messages_last_60s', 'repeated_message_60s', 'message_interval', 'gold', 'xp', 'last_hits', 'recent_teamfight_120s', 'recent_teamfight_loss_120s', 'sampling_group', 'label', 'notes']


,annotation_id,match_id,time,key
0,1,24301,1060,man ur life must be shit
1,2,49193,2180,dont know why
2,3,4367,2072,can u end
3,4,1909,2191,noob jugger
4,5,16069,1889,it is not about how bad he is


In [95]:
print(
    "Unique match-time-message combinations:",
    reviewed_labels[
        ["match_id", "time", "key"]
    ]
    .drop_duplicates()
    .shape[0]
)

Unique match-time-message combinations: 300


## 14. Verify Pilot Matches

确认已经标注的 300 条消息在完整数据中能够唯一匹配，避免下一轮抽样时重复选择已经标注的消息。

In [96]:
pilot_keys = (
    reviewed_labels[
        ["match_id", "time", "key"]
    ]
    .drop_duplicates()
)

pilot_matches = label_data.merge(
    pilot_keys,
    on=["match_id", "time", "key"],
    how="inner"
)

print("Pilot unique keys:")
print(len(pilot_keys))

print("\nMatching rows in full dataset:")
print(len(pilot_matches))

Pilot unique keys:
300

Matching rows in full dataset:
421


## 15. Create Additional Annotation Sample

从完整数据中排除已经标注的 300 条消息，再抽取 900 条新样本。

新样本包括：

- 300 random messages
- 300 emotional candidates
- 300 harmful candidates

In [97]:
existing_keys = set(
    zip(
        reviewed_labels["match_id"],
        reviewed_labels["time"],
        reviewed_labels["key"]
    )
)

available_data = label_data[
    ~label_data.apply(
        lambda row: (
            row["match_id"],
            row["time"],
            row["key"]
        ) in existing_keys,
        axis=1
    )
].copy()

print("Available rows:")
print(len(available_data))

Available rows:
1360960


In [98]:
RANDOM_STATE = 339

random_extra = (
    available_data
    .sample(
        n=300,
        random_state=RANDOM_STATE
    )
    .copy()
)

random_extra["sampling_group"] = "random"


remaining = available_data.drop(
    index=random_extra.index
)

emotional_pool = remaining[
    remaining["emotional_candidate"]
    &
    ~remaining["harmful_candidate"]
]

emotional_extra = (
    emotional_pool
    .sample(
        n=300,
        random_state=RANDOM_STATE
    )
    .copy()
)

emotional_extra["sampling_group"] = "emotional_candidate"


remaining = remaining.drop(
    index=emotional_extra.index
)

harmful_pool = remaining[
    remaining["harmful_candidate"]
]

harmful_extra = (
    harmful_pool
    .sample(
        n=300,
        random_state=RANDOM_STATE
    )
    .copy()
)

harmful_extra["sampling_group"] = "harmful_candidate"


additional_sample = pd.concat(
    [
        random_extra,
        emotional_extra,
        harmful_extra
    ],
    ignore_index=True
)

additional_sample = (
    additional_sample
    .sample(
        frac=1,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)

additional_sample["annotation_id"] = range(
    301,
    301 + len(additional_sample)
)

print("Additional sample size:")
print(len(additional_sample))

print("\nSampling groups:")
print(
    additional_sample[
        "sampling_group"
    ].value_counts()
)

Additional sample size:
900

Sampling groups:
sampling_group
random                 300
emotional_candidate    300
harmful_candidate      300
Name: count, dtype: int64


## 16. Prepare Additional Annotation File

整理新增的 900 条消息，保留人工标注需要的文本和上下文信息，并添加空白标签列。

In [99]:
annotation_columns = [
    "annotation_id",
    "match_id",
    "time",
    "key",
    "chat_context",

    "messages_last_60s",
    "repeated_message_60s",
    "message_interval",

    "gold",
    "xp",
    "last_hits",

    "recent_teamfight_120s",
    "recent_teamfight_loss_120s",

    "sampling_group"
]

additional_annotation_df = (
    additional_sample[
        annotation_columns
    ]
    .copy()
)

additional_annotation_df["label"] = ""
additional_annotation_df["notes"] = ""

print("Rows:")
print(len(additional_annotation_df))

display(
    additional_annotation_df.head(20)
)

Rows:
900


,annotation_id,match_id,time,key,chat_context,messages_last_60s,repeated_message_60s,message_interval,gold,xp,last_hits,recent_teamfight_120s,recent_teamfight_loss_120s,sampling_group,label,notes
0,301,45128,1610,end,; | :) | lol | just end it | утв,1,0,1.0,11486,13538,121,0,0,random,,
1,302,7082,351,why not little boy,man 5.15. | no fly.. | ? | ^^ | and why u crow...,0,0,24.0,2215,2563,22,0,0,emotional_candidate,,
2,303,15825,2164,fuck right outta here bounty,level 4 dagon cummin at ya | cool hero | gg | ...,1,0,71.0,9291,11805,28,0,0,random,,
3,304,28081,659,trash,OG | ahahahhaha | hahaah | more | OMG,0,0,118.0,4746,5359,38,0,0,harmful_candidate,,
4,305,45408,2364,bad game,4/6 feed :v | everyone on this team fed yiu | ...,0,0,45.0,9620,12977,56,1,1,emotional_candidate,,
5,306,1970,931,"this ember is so bad, my head hurts",loL | report mirana please | Ez mid,3,0,NaN,3034,2715,10,0,0,emotional_candidate,,
6,307,10197,1946,noob,forgot it? | my first blood? | dont remember? ...,9,0,1.0,19731,20433,175,1,0,harmful_candidate,,
7,308,46667,1417,Sad,kako bolje xd | ti mislis da ce pobjedis ? | n...,2,0,9.0,8837,9342,71,0,0,emotional_candidate,,
8,309,5472,591,so so so fucking bad,PUNCH | HIM | U FUCKING NOOB | pls report tusk...,1,0,8.0,1429,893,2,0,0,emotional_candidate,,
9,310,26149,2069,loss mrr ember haha,NICE GAME | fail mid | boy ks 8- 10 | its ok |...,0,0,7.0,13715,11863,106,1,1,random,,


## 17. Export Additional Annotation File

将新增的 900 条样本导出为 CSV，用于第二轮人工三分类标注。

In [100]:
additional_path = (
    LABEL_DIR
    / "annotation_additional_900.csv"
)

additional_annotation_df.to_csv(
    additional_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved to:")
print(additional_path)

Saved to:
D:\JupyterProject\AIML339_Toxicity\data\labels\annotation_additional_900.csv


## 18. Verify Additional Annotation File

重新读取新增的 900 条样本，确认编号、数量和标签列都正确。

In [101]:
check_additional = pd.read_csv(
    additional_path
)

print("Rows:")
print(len(check_additional))

print("\nAnnotation ID range:")
print(
    check_additional["annotation_id"].min(),
    "to",
    check_additional["annotation_id"].max()
)

print("\nMissing labels:")
print(
    check_additional["label"].isna().sum()
)

print("\nSampling groups:")
print(
    check_additional["sampling_group"]
    .value_counts()
)

Rows:
900

Annotation ID range:
301 to 1200

Missing labels:
900

Sampling groups:
sampling_group
random                 300
emotional_candidate    300
harmful_candidate      300
Name: count, dtype: int64


## 19. Load Additional Labels

读取新增的 900 条已标注数据，并检查所有标签是否完整有效。

In [102]:
ADDITIONAL_LABEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "labels"
    / "annotation_additional_900_labeled.csv"
)

additional_labels = pd.read_csv(
    ADDITIONAL_LABEL_PATH
)

print("Rows:")
print(len(additional_labels))

print("\nMissing labels:")
print(additional_labels["label"].isna().sum())

print("\nValid labels only:")
print(
    additional_labels["label"]
    .isin([0, 1, 2])
    .all()
)

print("\nLabel distribution:")
print(
    additional_labels["label"]
    .value_counts()
    .sort_index()
)

Rows:
900

Missing labels:
0

Valid labels only:
True

Label distribution:
label
0    279
1    344
2    277
Name: count, dtype: int64


## 20. Combine All Labels

将前 300 条 pilot 标签和新增的 900 条标签合并，形成完整的 1200 条标注数据。

In [103]:
all_labels = pd.concat(
    [
        reviewed_labels,
        additional_labels
    ],
    ignore_index=True
)

all_labels = (
    all_labels
    .sort_values("annotation_id")
    .reset_index(drop=True)
)

print("Total labeled messages:")
print(len(all_labels))

print("\nAnnotation ID range:")
print(
    all_labels["annotation_id"].min(),
    "to",
    all_labels["annotation_id"].max()
)

print("\nLabel distribution:")
print(
    all_labels["label"]
    .value_counts()
    .sort_index()
)

Total labeled messages:
1200

Annotation ID range:
1 to 1200

Label distribution:
label
0    396
1    462
2    342
Name: count, dtype: int64


## 21. Check Duplicate Messages

检查 1200 条标注消息是否具有唯一的比赛、时间和消息组合，避免标签匹配到错误记录。

In [104]:
unique_keys = (
    all_labels[
        ["match_id", "time", "key"]
    ]
    .drop_duplicates()
)

print("Total labeled rows:")
print(len(all_labels))

print("\nUnique match-time-message combinations:")
print(len(unique_keys))

Total labeled rows:
1200

Unique match-time-message combinations:
1200


## 22. Build Final Labeled Dataset

直接使用之前实际抽取的 1200 条样本，并通过唯一的 annotation ID 加入最终标签，避免相同时间和相同文本造成重复匹配。

In [105]:
selected_samples = pd.concat(
    [
        pilot,
        additional_sample
    ],
    ignore_index=True
)

print("Selected samples:")
print(len(selected_samples))

print("\nUnique annotation IDs:")
print(
    selected_samples["annotation_id"]
    .nunique()
)

Selected samples:
1200

Unique annotation IDs:
1200


In [106]:
label_lookup = all_labels[
    [
        "annotation_id",
        "label",
        "notes"
    ]
].copy()

final_labeled_data = selected_samples.merge(
    label_lookup,
    on="annotation_id",
    how="left",
    validate="one_to_one"
)

print("Final labeled rows:")
print(len(final_labeled_data))

print("\nMissing labels:")
print(
    final_labeled_data["label"]
    .isna()
    .sum()
)

print("\nLabel distribution:")
print(
    final_labeled_data["label"]
    .value_counts()
    .sort_index()
)

Final labeled rows:
1200

Missing labels:
0

Label distribution:
label
0    396
1    462
2    342
Name: count, dtype: int64


## 23. Remove Player Names

删除玩家显示名称，避免在最终训练数据中保留可识别信息。比赛 ID 和玩家位置仅用于数据划分和分析，不作为文本模型输入。

In [107]:
if "unit" in final_labeled_data.columns:
    final_labeled_data = (
        final_labeled_data
        .drop(columns=["unit"])
    )

print("Final shape:")
print(final_labeled_data.shape)

print("\nColumns:")
print(
    final_labeled_data.columns.tolist()
)

Final shape:
(1200, 26)

Columns:
['match_id', 'time', 'player_slot', 'key', 'prev_message_1', 'prev_message_2', 'prev_message_3', 'prev_message_4', 'prev_message_5', 'gold', 'xp', 'last_hits', 'messages_last_60s', 'repeated_message_60s', 'message_interval', 'player_team', 'seconds_since_teamfight', 'recent_teamfight_120s', 'recent_teamfight_loss_120s', 'chat_context', 'emotional_candidate', 'harmful_candidate', 'sampling_group', 'annotation_id', 'label', 'notes']


## 24. Save Final Labeled Dataset

保存最终的 1200 条三分类数据，供 baseline 和 context-aware 模型训练使用。

In [108]:
FINAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "labeled_context_1200.parquet"
)

final_labeled_data.to_parquet(
    FINAL_PATH,
    index=False
)

print("Saved to:")
print(FINAL_PATH)

print("\nSaved rows:")
print(len(final_labeled_data))

Saved to:
D:\JupyterProject\AIML339_Toxicity\data\processed\labeled_context_1200.parquet

Saved rows:
1200


## 25. Verify Final Dataset

重新读取最终数据，确认保存后的行数和标签均正确。

In [109]:
check_final = pd.read_parquet(
    FINAL_PATH
)

print("Loaded rows:")
print(len(check_final))

print("\nMissing labels:")
print(
    check_final["label"]
    .isna()
    .sum()
)

print("\nLabel distribution:")
print(
    check_final["label"]
    .value_counts()
    .sort_index()
)

Loaded rows:
1200

Missing labels:
0

Label distribution:
label
0    396
1    462
2    342
Name: count, dtype: int64
